In [ ]:
def run_genetic_cross(
    ancestors_poAncestry_1: List['Individual'],
    ancestors_poAncestry_2: List['Individual'],
    offspring_count_per_mating_pair: int,
    generation_label: str,
    num_chromosomes_for_offspring: int,
    recomb_event_probabilities: Dict[int, float],
    recomb_probabilities: List[float]
) -> List['Individual']:
    """
    Simulates a genetic cross between individuals from two parental populations to produce a new generation of offspring.

    This function pairs individuals from `ancestors_poAncestry_1` and `ancestors_poAncestry_2`
    and for each unique mating pair, generates a specified number of offspring.
    The genetic material for each offspring is created by simulating meiosis with recombination
    for each chromosome pair, combining one recombinant chromatid from each parent.
    All generated offspring's genetic and recombination data are recorded.

    Args:
        ancestors_poAncestry_1 (List[Individual]): The first group of parental individuals available for mating.
        ancestors_poAncestry_2 (List[Individual]): The second group of parental individuals available for mating.
        offspring_count_per_mating_pair (int): The number of new offspring individuals to generate
                                               *from each unique mating pair*.
        generation_label (str): A descriptive label for the new generation being created (e.g., "F2", "BC1_A").
                                This label is used for data recording.
        num_chromosomes_for_offspring (int): The number of diploid chromosome pairs each new offspring will inherit.
                                             This should typically match the parents' chromosome count.
        recomb_event_probabilities (dict): A probability distribution (e.g., {0: p0, 1: p1, 2: p2}) that dictates
                                           the likelihood of 0, 1, 2, or more recombination events occurring
                                           on a single chromosome during meiosis.
        recomb_probabilities (List[float]): A list of probabilities for recombination occurring at each
                                            specific interval *between* loci along a chromosome. Its length
                                            should be `num_loci_per_chromosome - 1`.

    Returns:
        List[Individual]: A list containing all the newly created offspring individuals from this cross.
    """
    # Debug prints (useful for monitoring the simulation flow and population sizes)
    print(f"\nDEBUG_CROSS for {generation_label}")
    print(f"DEBUG_CROSS: Parent A size entering cross: {len(ancestors_poAncestry_1)}")
    print(f"DEBUG_CROSS: Parent B size entering cross: {len(ancestors_poAncestry_2)}")
    print(f"DEBUG_CROSS: Offspring *per mating pair* expected: {offspring_count_per_mating_pair}")

    progeny = [] # Initialize an empty list to store all the new offspring individuals.

    # Shuffle the parental populations randomly. This is crucial to ensure random mating
    # and to allow for unique pairing without replacement when using `zip`.
    shuffled_ancestor_A = random.sample(ancestors_poAncestry_1, len(ancestors_poAncestry_1))
    shuffled_ancestor_B = random.sample(ancestors_poAncestry_2, len(ancestors_poAncestry_2))

    # Determine the actual number of unique mating pairs that can be formed.
    # This is limited by the size of the smaller parental population.
    num_mating_pairs = min(len(shuffled_ancestor_A), len(shuffled_ancestor_B))
    print(f"DEBUG_CROSS: Number of unique mating pairs formed: {num_mating_pairs}")

    # Iterate through unique pairs of parents using `zip` on the shuffled lists.
    # Each iteration represents one mating event.
    for ancestor_A, ancestor_B in zip(shuffled_ancestor_A, shuffled_ancestor_B):
        # For EACH unique mating pair, generate the specified number of offspring.
        # This inner loop controls the family size for each cross.
        for _ in range(offspring_count_per_mating_pair):
            # Create a new 'Individual' instance for the offspring.
            # The number of loci per chromosome is assumed to be consistent across generations,
            # so it's taken from ancestor_A.
            offspring = Individual(
                num_chromosomes=num_chromosomes_for_offspring,
                num_loci_per_chromosome=ancestor_A.num_loci_per_chromosome
            )

            # Now, populate the offspring's chromosomes by simulating inheritance from parents.
            # This loop runs for each chromosome pair the offspring will have.
            for chr_idx in range(num_chromosomes_for_offspring):
                # Get the relevant diploid chromosome pair from each parent for the current chromosome index.
                diploid_pair_A = ancestor_A.diploid_chromosome_pairs[chr_idx]
                diploid_pair_B = ancestor_B.diploid_chromosome_pairs[chr_idx]

                # Simulate meiosis with recombination to get one haploid chromatid (gamete) from each parent.
                # `meiosis_with_recombination` handles the crossover events and generates the resulting chromatid.
                haploid_from_A = meiosis_with_recombination(diploid_pair_A, recomb_event_probabilities, recomb_probabilities)
                haploid_from_B = meiosis_with_recombination(diploid_pair_B, recomb_event_probabilities, recomb_probabilities)

                # Combine these two haploid chromatids to form a new diploid chromosome pair for the offspring.
                # One chromatid comes from parent A, the other from parent B.
                offspring.diploid_chromosome_pairs.append(DiploidChromosomePair(haploid_from_A, haploid_from_B))

            # Record the newly created offspring's full genetic data and recombination patterns.
            # These functions add detailed records to global data structures.
            record_individual_genome(offspring, generation_label)
            record_chromatid_recombination(offspring, generation_label)

            # Add the fully constructed and recorded offspring to the list of progeny for this cross.
            progeny.append(offspring)

    print(f"DEBUG_CROSS: Final new_generation size created: {len(progeny)}")
    print(f"END DEBUG_CROSS for {generation_label}\n")

    return progeny # Return the complete list of all offspring generated from this cross.